In [ ]:
import pymysql

conn = pymysql.connect(host='localhost', user='root', password='1111', database='finance',
                       charset='utf8'
                    #  autocommit=False # default 값 : False
                       )
cur = conn.cursor()

In [ ]:
from decimal import Decimal, InvalidOperation
from datetime import datetime, date
from pathlib import Path
from dataclasses import dataclass
import csv
import re
import pandas as pd
from IPython.display import display

# 프로젝트 root 찾는 기능 / 자료 파일 위치 확인
# 현재폴더, 현재 폴더의 상위 폴더들, 현재 폴더 아래에 있는 finance-sql 폴더
_candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'finance-sql']
ROOT = next((p for p in _candidates if (p / 'data/statement.csv').is_file()
             and (p / 'finance_helpers.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('전체 ZIP을 풀고 finance-sql 폴더에서 Jupyter를 시작하세요.')

In [15]:
print('연결 타입:', type(conn))
print('커서 타입:', type(cur))
print('자동 커밋:', conn.get_autocommit())  # 자동 커밋
assert type(cur) is pymysql.cursors.Cursor
assert conn.get_autocommit() is False

연결 타입: <class 'pymysql.connections.Connection'>
커서 타입: <class 'pymysql.cursors.Cursor'>
자동 커밋: False


* 서버 버전·현재 데이터베이스 확인

In [ ]:
cur.execute('SELECT VERSION()')  # MySQL 서버의 버전
version_row = cur.fetchone()
print('VERSION 행:', version_row, '행 타입:', type(version_row))
print('버전 값:', version_row[0])

cur.execute('SELECT DATABASE()')
database_name = cur.fetchone()[0]
print('현재 DB:', database_name)
assert database_name == 'sqldb'

VERSION 행: ('8.0.46',) 행 타입: <class 'tuple'>
버전 값: 8.0.46
현재 DB: sqldb


* 초기 금융 데이터의 기준선
    * 계좌 정보와 거래 원장(fin_ledger_entries)의 데이터가 예상한 결과와 일치하는지 검증
    * 거래 원장 : 계좌에서 발생한 모든 돈의 시간순 이동 기록

In [19]:
# 계좌 정보 조회
cur.execute('SELECT account_id, balance FROM fin_accounts ORDER BY account_id')
accounts = cur.fetchall()
print('계좌:', accounts)

# 계좌 잔액 검증
assert accounts == ((101, Decimal('1120000.00')),
                    (102, Decimal('590000.00')),
                    (103, Decimal('190000.00')))

# 계좌 개수 검증
cur.execute('SELECT COUNT(*) FROM fin_accounts')
assert cur.fetchone()[0] == 3

# 거래 원장 전체 건수 조회
cur.execute('SELECT COUNT(*) FROM fin_ledger_entries')
ledger_count = cur.fetchone()[0]
assert ledger_count == 9

# 거래 상태별 건수 집계
cur.execute('SELECT status, COUNT(*) FROM fin_ledger_entries GROUP BY status ORDER BY status')
status_rows = cur.fetchall()
assert ledger_count == 9
assert dict(status_rows) == {'posted': 7, 'pending': 1, 'reversed': 1}

print('원장 총수:', ledger_count, '상태별:', status_rows)


계좌: ((101, Decimal('1120000.00')), (102, Decimal('590000.00')), (103, Decimal('190000.00')))
원장 총수: 9 상태별: (('pending', 1), ('posted', 7), ('reversed', 1))


* 컬럼 이름, 조회 결과의 자료형, 금액의 자료형, 딕셔너리 변환, 조회 결과를 다 사용했는지에 대한 여부

In [23]:
cur.execute('SELECT account_id, currency, balance FROM fin_accounts WHERE account_id=%s', (101,))
# print(cur.description)
column_names = [item[0] for item in cur.description]
row = cur.fetchone()
print('열 이름:', column_names)
print('행:', row, '행 타입:', type(row))
print('잔액:', row[2], '금액 타입:', type(row[2]))
print('설명용 이름 매핑:', dict(zip(column_names, row)))
assert isinstance(row, tuple)
assert isinstance(row[2], Decimal)
assert cur.fetchone() is None

열 이름: ['account_id', 'currency', 'balance']
행: (101, 'KRW', Decimal('1120000.00')) 행 타입: <class 'tuple'>
잔액: 1120000.00 금액 타입: <class 'decimal.Decimal'>
설명용 이름 매핑: {'account_id': 101, 'currency': 'KRW', 'balance': Decimal('1120000.00')}


* fetchone: 한 행씩 소비

In [24]:
account_sql = 'SELECT account_id FROM fin_accounts ORDER BY account_id'
cur.execute(account_sql)
first = cur.fetchone()
second = cur.fetchone()
remaining = cur.fetchall()

print('첫째:', first, '둘째:', second, '나머지:', remaining)


첫째: (101,) 둘째: (102,) 나머지: ((103,),)


* fetchmany: 지정한 수만큼 묶기

In [27]:
cur.execute(account_sql)  # 재조회: 결과 읽기 위치를 처음으로
page1 = cur.fetchmany(2)
page2 = cur.fetchmany(2)
page3 = cur.fetchmany(2)

print('1묶음:', page1, '2묶음:', page2, '3묶음:', page3)

assert page1 == ((101,), (102,))
assert page2 == ((103,),)
assert not page3

1묶음: ((101,), (102,)) 2묶음: ((103,),) 3묶음: ()


* fetchall: 현재 위치부터 남은 전부

In [ ]:
cur.execute(account_sql)
all_accounts = cur.fetchall()
empty_after_all = cur.fetchall()
print('전체:', all_accounts, '소진 후:', empty_after_all)

* 자동 커밋·문자셋·안전 경계

In [28]:
# @@autocommit: 현재 세션의 자동 커밋 설정
# @@character_set_client: 클라이언트 문자 집합
# @@character_set_connection: 연결 문자 집합

cur.execute('SELECT @@autocommit, @@character_set_client, @@character_set_connection')
settings = cur.fetchone()
print('서버 세션 설정:', settings)

서버 세션 설정: (0, 'utf8mb3', 'utf8mb3')


* test code

In [ ]:
# # test code
# numbers = [10, 20, 30, 40]

# result = (n for n in numbers if n >= 30)

# print(next(result))
# print(next(result))

30
40


In [ ]:
# # test code
# numbers = [10, 20, 30, 40]

# result = next(
#     (n for n in numbers if n >= 30)
# )

# print(result)

30


In [ ]:
# test code
# for p in _candidates:
#     if (p / 'data/statement.csv').is_file() and (p / 'finance_helpers.py').is_file()
#         return True
#     else:
#         None

In [ ]:
# test code
# Path.cwd()
# ####
# for p in Path.cwd().parents:
#     print(p)
# ####
# numbers = [1, 2, 3]

# print(numbers)
# print(*numbers)
# ###
# numbers = [0, *[1, 2, 3], 4]

# print(numbers)
# ###
# print(*Path.cwd().parents)
# ###
# _candidates

WindowsPath('c:/Project/04.데이터저장/Src/finance-sql')

[WindowsPath('c:/Project/04.데이터저장/Src/finance-sql'),
 WindowsPath('c:/Project/04.데이터저장/Src'),
 WindowsPath('c:/Project/04.데이터저장'),
 WindowsPath('c:/Project'),
 WindowsPath('c:/'),
 WindowsPath('c:/Project/04.데이터저장/Src/finance-sql/finance-sql')]